## Week 5: Hybrid Search + Re-ranking


| 실험 구성 | 내용 |
|-----------|------|
| Baseline | Week 4 최적: 섹션분리 + Recursive (chunk=1000, overlap=150) |
| Hybrid Search | BM25(sparse) + Dense(OpenAI) + RRF |
| Re-ranking | BAAI/bge-reranker-v2-m3 Cross-Encoder |

---


In [37]:
import os
import sys
import re

sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import CrossEncoder

from utils.doc_preprocessing import extract_sections, get_breadcrumb

load_dotenv("../.env")

for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        matplotlib.rc("font", family=font)
        break
    except Exception:
        pass
matplotlib.rcParams["axes.unicode_minus"] = False

print("라이브러리 로드 완료")

라이브러리 로드 완료



### 1. Week 4 최적 Chunking 전략

**선택 전략**: 섹션분리 + Recursive Chunking (chunk_size=1000, overlap=150)


In [41]:
PDF_PATH = "../data/registration_of_real_estatee_manual.pdf"

sections, all_headings = extract_sections(PDF_PATH)

section_docs = [
    Document(
        page_content=re.sub(r'\n+', ' ', s["content"]).strip(),
        metadata={
            "section_num": s["num"],
            "section_title": s["title"],
            "breadcrumb": get_breadcrumb(s["num"], all_headings),
            "start_page": s["start_page"],
        },
    )
    for s in sections
    if s["content"].strip()
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = [c for c in text_splitter.split_documents(section_docs) if c.page_content.strip()]

print(f"섹션 수     : {len(sections)}")
print(f"총 청크 수  : {len(chunks)}")
print(f"평균 청크 길이: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

섹션 수     : 86
총 청크 수  : 250
평균 청크 길이: 767 chars


### 2. Dense Retriever (Baseline)

**모델**: OpenAI `text-embedding-3-large`  
**벡터 DB**: ChromaDB (MMR 검색, k=5)


In [42]:
DENSE_DB_PATH = "../chroma_db/real_estate_RAG"

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(DENSE_DB_PATH) and os.listdir(DENSE_DB_PATH):
    print("기존 Chroma DB 로드 중...")
    db = Chroma(
        persist_directory=DENSE_DB_PATH,
        embedding_function=embeddings,
        collection_name="real_estate_RAG",
    )
else:
    print("Chroma DB 생성 중... (OpenAI embedding API 호출)")
    db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DENSE_DB_PATH,
        collection_name="real_estate_RAG",
    )

dense_retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20},
)

print(f"Dense Retriever 준비 완료")
print(f"  - 인덱싱 청크 수: {db._collection.count()}")
print(f"  - 검색 방식: MMR (k=5, fetch_k=20)")

기존 Chroma DB 로드 중...
Dense Retriever 준비 완료
  - 인덱싱 청크 수: 250
  - 검색 방식: MMR (k=5, fetch_k=20)



### 3. BM25 Retriever (Sparse)

In [43]:
def korean_tokenizer(text: str):
    """BM25용 한국어 토크나이저: 특수문자 제거 + 공백 분할 + 1글자 필터"""
    cleaned = re.sub("[^가-힣a-zA-Z0-9]", " ", text)
    return [t for t in cleaned.split() if len(t) > 1]

# 토크나이저 동작 확인
sample = "근저당권 말소등기 신청 시 필요한 서류"
print(f"입력: {sample}")
print(f"토큰: {korean_tokenizer(sample)}")
print()

bm25_retriever = BM25Retriever.from_documents(
    chunks,
    k=5,
    preprocess_func=korean_tokenizer,
)

print(f"BM25 Retriever 준비 완료")
print(f"  - 인덱싱 청크 수: {len(chunks)}")
print(f"  - 토크나이저: korean_tokenizer")

입력: 근저당권 말소등기 신청 시 필요한 서류
토큰: ['근저당권', '말소등기', '신청', '필요한', '서류']

BM25 Retriever 준비 완료
  - 인덱싱 청크 수: 250
  - 토크나이저: korean_tokenizer



### 4. Hybrid Search (BM25 + Dense + RRF)

In [44]:
# k=5: Ablation 비교용
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
    c=60,
)

print("Hybrid Retriever (BM25 + Dense + RRF) 준비 완료")
print("  - 가중치: BM25=0.5, Dense=0.5")
print("  - 융합 방식: Reciprocal Rank Fusion (c=60)")
print("  - 가중치 0.5:0.5 선택 이유: 법률 도메인에서 키워드/의미 균형이 중요")

Hybrid Retriever (BM25 + Dense + RRF) 준비 완료
  - 가중치: BM25=0.5, Dense=0.5
  - 융합 방식: Reciprocal Rank Fusion (c=60)
  - 가중치 0.5:0.5 선택 이유: 법률 도메인에서 키워드/의미 균형이 중요


In [45]:
# k=20: Re-ranker의 1차 후보 확보용
bm25_retriever_k20 = BM25Retriever.from_documents(
    chunks,
    k=20,
    preprocess_func=korean_tokenizer,
)
dense_retriever_k20 = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 20, "fetch_k": 50},
)
hybrid_retriever_k20 = EnsembleRetriever(
    retrievers=[bm25_retriever_k20, dense_retriever_k20],
    weights=[0.5, 0.5],
    c=60,
)

print("Hybrid k=20 Retriever 준비 완료 (Re-ranker 1차 후보용)")

Hybrid k=20 Retriever 준비 완료 (Re-ranker 1차 후보용)


---
### 5. Cross-Encoder Re-ranker

| 모델 | 언어 지원 | 한국어 성능 | 크기 | 선택 |
|------|-----------|-------------|------|------|
| `BAAI/bge-reranker-v2-m3` | 다국어 | 강함 | 중간 | **선택** |
| `dragonkue/bge-reranker-v2-m3-ko` | 한국어 특화 | 매우 강함 | 중간 | 차선 |
| `cross-encoder/ms-marco-MiniLM-L6-v2` | 영어 중심 | 약함 | 가벼움 | 미선택 |

**선택 이유**: 부동산 등기 문서에는 한국어 법률 용어와 법령 번호(영문/숫자 혼합)가 혼재한다. `bge-reranker-v2-m3`는 다국어 corpus로 훈련되어 한국어/영문 혼합 쿼리에서 안정적 성능을 보이며, 한국어 특화 버전과 성능 차이가 작은 대신 일반화 능력이 높다.

### 파이프라인
```
질문 → Hybrid Retriever (top-20 후보) → Cross-Encoder (재정렬) → top-5 반환
```

In [46]:
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
print(f"Cross-Encoder 모델 로드 중: {RERANKER_MODEL}")
print("(첫 실행 시 모델 다운로드 필요 ~1.1GB)")

cross_encoder = CrossEncoder(RERANKER_MODEL)
print("Cross-Encoder Re-ranker 로드 완료")


def rerank(query: str, docs: list, top_k: int = 5) -> list:
    """Cross-Encoder로 문서 목록 재정렬 후 top_k 반환"""
    if not docs:
        return docs
    pairs = [(query, doc.page_content) for doc in docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]


def hybrid_rerank_retriever(query: str, top_k: int = 5) -> list:
    """Hybrid top-20 1차 검색 → Cross-Encoder top-5 재정렬"""
    candidates = hybrid_retriever_k20.invoke(query)
    return rerank(query, candidates, top_k=top_k)


print("Hybrid + Re-ranker 파이프라인 준비 완료")

Cross-Encoder 모델 로드 중: BAAI/bge-reranker-v2-m3
(첫 실행 시 모델 다운로드 필요 ~1.1GB)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 672.46it/s]


Cross-Encoder Re-ranker 로드 완료
Hybrid + Re-ranker 파이프라인 준비 완료


### 6. 정성 평가 (Qualitative Comparison)


In [ ]:
TEST_QUESTIONS = [
    "토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?",
    "근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?",
    "부동산 매매로 인한 소유권 이전등기 절차를 설명해줘",
    "전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?",
    "상속으로 인한 부동산 등기 신청 시 필요한 서류는 무엇인가요?",
]

테스트 질문 세트: 5개
  Q1: 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?
  Q2: 근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?
  Q3: 부동산 매매로 인한 소유권 이전등기 절차를 설명해줘
  Q4: 전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?
  Q5: 상속으로 인한 부동산 등기 신청 시 필요한 서류는 무엇인가요?


In [20]:
def compare_retrievers(query: str, k: int = 5):
    """4가지 retriever 결과를 나란히 출력"""
    results = {
        "BM25": bm25_retriever.invoke(query)[:k],
        "Dense (MMR)": dense_retriever.invoke(query)[:k],
        "Hybrid (BM25+Dense+RRF)": hybrid_retriever.invoke(query)[:k],
        "Hybrid + Re-ranker": hybrid_rerank_retriever(query, top_k=k),
    }

    print(f"\n{'='*72}")
    print(f"질문: {query}")
    print(f"{'='*72}")

    for method, docs in results.items():
        print(f"\n[{method}]")
        for i, doc in enumerate(docs, 1):
            breadcrumb = doc.metadata.get("breadcrumb", "N/A")
            page = doc.metadata.get("start_page", "?")
            preview = doc.page_content.replace("\n", " ")[:100]
            print(f"  {i}. p.{page} [{breadcrumb}]")
            print(f"     {preview}...")

In [16]:
# Q1: 토지 소유권 보존등기 (Week 4에서 실패 경험)
compare_retrievers(TEST_QUESTIONS[0])


질문: 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?

[BM25]
  1. p.15 [소유권보존등기 > 토지소유권보존등기 > 신청서작성]
     토지소유권보존등기신청서 신청서및첨부서류의순서 신청서 , 취득세 ( 등록면허세 ) 영수필확인서 , 등기신청수수료영수필확인서 , 위임장 , 주민등록표초 ( 등 ) 본 , 토지 ( 임야...
  2. p.29 [소유권보존등기 > 구분건물소유권보존등기 > 개념및신청인]
     구분건물소유권보존등기의개념 구분건물소유권보존등기의개념 " 구분건물 소유권 보존등기 " 란 소유자의 신청에 의해미등기의구분건물에 처음으로행해지는소유권 등기를말합니다 ( 대법원인터넷등...
  3. p.20 [소유권보존등기 > 건물소유권보존등기 > 개념및신청인]
     건물소유권보존등기의개념 건물소유권보존등기의개념 " 건물 소유권 보존등기 " 란 소유자의 신청에 의해 미등기의 건물에 처음으로행해지는소유권등기를 말합니다 ( 대법원인터넷등기소 - 자...
  4. p.199 [가등기 > 가등기에기한소유권이전본등기 > 제출서류]
     . ※ 시 , 군 , 구청세무과를방문해취득세납부고지서를발부받고세금을은행에서납부하면됩니다 . 은행을통해준비해야할서류 취득세영수필확인서 시 · 군 · 구청 세무과에서 취득세납부고지서를...
  5. p.199 [가등기 > 가등기에기한소유권이전본등기 > 제출서류]
     . 매매가가 1 억 5 천만원인이 건물은 매입기준의 주택 및 토지 외의 부동산 시가표준액 1 억 3 천만원 이상 2 억 5 천만원미만에 해당하고 기타 지역인 수원이므로 , 주택 매...

[Dense (MMR)]
  1. p.10 [소유권보존등기 > 토지소유권보존등기 > 개념및신청인]
     토지소유권보존등기의개념 토지소유권보존등기의개념 " 토지소유권보존등기 " 란토지소유자의신청에의해미등기의토지에처음으로행해지는소유권등기를 말합니다 ( 출처 : 대한민국법원인터넷등기소 홈...
  2. p.15 [소유권보존등기 > 토지소유권보존등기 > 신청서작성]
 

In [ ]:
# Q2: 근저당권 말소등기
compare_retrievers(TEST_QUESTIONS[1])

In [ ]:
# Q3: 소유권 이전등기
compare_retrievers(TEST_QUESTIONS[2])

In [ ]:
# Q4: 전세권 vs 임차권
compare_retrievers(TEST_QUESTIONS[3])

In [ ]:
# Q5: 상속 등기
compare_retrievers(TEST_QUESTIONS[4])

---
### 7. RAGAS Ablation 비교 실험

### 평가 지표 (RAGAS)
- **Faithfulness**: 생성 답변이 검색 문서에 기반하는가 (환각 탐지)
- **Answer Relevancy**: 답변이 질문과 관련 있는가
- **Context Precision**: 검색된 문서 중 관련 문서 비율

**Judge LLM**: Gemini 2.5 Flash (비용 효율 + 한국어 이해)

In [21]:
import os
os.environ["RAGAS_MAX_CONCURRENCY"] = "1"

from langchain_google_genai import ChatGoogleGenerativeAI
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithoutReference,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# RAG 공통 LLM + Prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """당신은 대한민국 부동산 등기 업무 전문가입니다.
제공된 [참고 문서]의 내용을 바탕으로 사용자의 질문에 사실에 근거하여 답변하세요.

지침:
1. 법률적 근거가 문서에 명시되어 있다면 반드시 포함하세요.
2. 문서 내용으로 답변할 수 없는 경우, '확인 불가'라고 답변하세요.
3. 이해하기 쉽게 설명하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]"""
)


def get_answer(query: str, docs: list) -> str:
    """검색 문서 기반 RAG 답변 생성"""
    context = "\n\n".join(d.page_content for d in docs)
    messages = prompt.format_messages(context=context, question=query)
    return llm.invoke(messages).content


def build_ragas_samples(retriever_fn, questions: list) -> list:
    """retriever_fn(query) -> list[Document] 를 받아 RAGAS 샘플 생성"""
    samples = []
    for i, q in enumerate(questions, 1):
        print(f"  [{i}/{len(questions)}] {q[:50]}...")
        docs = retriever_fn(q)
        answer = get_answer(q, docs)
        samples.append(SingleTurnSample(
            user_input=q,
            response=answer,
            retrieved_contexts=[d.page_content for d in docs],
        ))
    return samples


# RAGAS Judge 설정
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        model_kwargs={"seed": 42},
    )
)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

ragas_metrics = [
    Faithfulness(),
    AnswerRelevancy(),
    LLMContextPrecisionWithoutReference(),
]

print("RAGAS 평가 환경 준비 완료")
print(f"  - Judge LLM: gemini-2.5-flash")
print(f"  - 평가 지표: Faithfulness, AnswerRelevancy, ContextPrecision")

RAGAS 평가 환경 준비 완료
  - Judge LLM: gemini-2.5-flash
  - 평가 지표: Faithfulness, AnswerRelevancy, ContextPrecision


In [23]:
# 4가지 Ablation 구성 정의
ABLATION_CONFIGS = {
    "BM25 Only": lambda q: bm25_retriever.invoke(q)[:5],
    "Dense Only (baseline)": lambda q: dense_retriever.invoke(q)[:5],
    "Hybrid (BM25+Dense+RRF)": lambda q: hybrid_retriever.invoke(q)[:5],
    "Hybrid + Re-ranker": lambda q: hybrid_rerank_retriever(q, top_k=5),
}

ablation_results = {}

for method_name, retriever_fn in ABLATION_CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"[{method_name}] 샘플 수집 중...")
    print(f"{'='*55}")

    samples = build_ragas_samples(retriever_fn, TEST_QUESTIONS)
    dataset = EvaluationDataset(samples=samples)

    print(f"  RAGAS 평가 실행 중...")
    result = evaluate(
        dataset=dataset,
        metrics=ragas_metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )

    df = result.to_pandas()
    ablation_results[method_name] = {
        "faithfulness": float(df["faithfulness"].mean()),
        "answer_relevancy": float(df["answer_relevancy"].mean()),
        "context_precision": float(df["llm_context_precision_without_reference"].mean()),
        "avg": float(
            df[["faithfulness", "answer_relevancy", "llm_context_precision_without_reference"]]
            .mean(axis=1).mean()
        ),
    }
    print(f"  완료: {ablation_results[method_name]}")

print("\n모든 Ablation 평가 완료")


[BM25 Only] 샘플 수집 중...
  [1/5] 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?...
  [2/5] 근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?...
  [3/5] 부동산 매매로 인한 소유권 이전등기 절차를 설명해줘...
  [4/5] 전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?...
  [5/5] 상속으로 인한 부동산 등기 신청 시 필요한 서류는 무엇인가요?...
  RAGAS 평가 실행 중...


Evaluating:   7%|▋         | 1/15 [00:03<00:46,  3.30s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 15/15 [01:05<00:00,  4.39s/it]


  완료: {'faithfulness': 0.6659692710035958, 'answer_relevancy': 0.6806298158693938, 'context_precision': 0.7499999999485001, 'avg': 0.6988663622738299}

[Dense Only (baseline)] 샘플 수집 중...
  [1/5] 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?...
  [2/5] 근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?...
  [3/5] 부동산 매매로 인한 소유권 이전등기 절차를 설명해줘...
  [4/5] 전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?...
  [5/5] 상속으로 인한 부동산 등기 신청 시 필요한 서류는 무엇인가요?...
  RAGAS 평가 실행 중...


Evaluating:   7%|▋         | 1/15 [00:03<00:47,  3.36s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  13%|█▎        | 2/15 [00:04<00:23,  1.82s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 15/15 [00:54<00:00,  3.65s/it]


  완료: {'faithfulness': 0.799008281573499, 'answer_relevancy': 0.973238641916074, 'context_precision': 0.7799999999406666, 'avg': 0.8507489744767465}

[Hybrid (BM25+Dense+RRF)] 샘플 수집 중...
  [1/5] 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?...
  [2/5] 근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?...
  [3/5] 부동산 매매로 인한 소유권 이전등기 절차를 설명해줘...
  [4/5] 전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?...
  [5/5] 상속으로 인한 부동산 등기 신청 시 필요한 서류는 무엇인가요?...
  RAGAS 평가 실행 중...


Evaluating:   7%|▋         | 1/15 [00:03<00:43,  3.13s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 15/15 [00:52<00:00,  3.50s/it]


  완료: {'faithfulness': 0.9049473684210525, 'answer_relevancy': 0.9741204366056448, 'context_precision': 0.5966666666218333, 'avg': 0.8252448238828436}

[Hybrid + Re-ranker] 샘플 수집 중...
  [1/5] 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?...
  [2/5] 근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?...
  [3/5] 부동산 매매로 인한 소유권 이전등기 절차를 설명해줘...
  [4/5] 전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?...
  [5/5] 상속으로 인한 부동산 등기 신청 시 필요한 서류는 무엇인가요?...
  RAGAS 평가 실행 중...


Evaluating:   7%|▋         | 1/15 [00:03<00:49,  3.53s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 15/15 [00:58<00:00,  3.89s/it]


  완료: {'faithfulness': 0.9178428093645487, 'answer_relevancy': 0.8439044877317963, 'context_precision': 0.8333333332988889, 'avg': 0.8650268767984113}

모든 Ablation 평가 완료


In [24]:
# 결과 테이블
results_df = pd.DataFrame(ablation_results).T
results_df.columns = ["Faithfulness", "Answer Relevancy", "Context Precision", "평균"]
results_df.index.name = "Retrieval 방법"
results_df = results_df.round(4)

print("=" * 65)
print("Ablation Study 결과")
print("=" * 65)
display(results_df)

# 최고 성능 방법 표시
best_method = results_df["평균"].idxmax()
print(f"\n최고 성능 방법: {best_method} (평균: {results_df.loc[best_method, '평균']:.4f})")

Ablation Study 결과


,Faithfulness,Answer Relevancy,Context Precision,평균
Retrieval 방법,,,,
BM25 Only,0.6660,0.6806,0.7500,0.6989
Dense Only (baseline),0.7990,0.9732,0.7800,0.8507
Hybrid (BM25+Dense+RRF),0.9049,0.9741,0.5967,0.8252
Hybrid + Re-ranker,0.9178,0.8439,0.8333,0.8650



최고 성능 방법: Hybrid + Re-ranker (평균: 0.8650)


### 8. Self-Query Retriever (메타데이터 자동 필터링)

### 핵심 아이디어

Week 4에서 설계한 `breadcrumb` 메타데이터를 **검색 필터로 활용**해 범위를 좁힌다.

```
질문: "토지 소유권 보존등기 제출서류 알려줘"
  ↓ LLM 또는 키워드 분석
필터: breadcrumb contains "소유권보존등기" AND section_title == "제출서류"
  ↓ Chroma where 필터 적용
Dense 검색 (필터된 청크 내에서만)
```

### 두 가지 구현 방식 비교

| 방식 | 원리 | 장점 | 단점 |
|------|------|------|------|
| **A. SelfQueryRetriever** | LLM이 질문 → 메타데이터 필터 자동 생성 | 완전 자동화, 복잡한 조건 처리 | LLM 비용, 필터 정확도 불확실 |
| **B. Manual BreadcrumbFilter** | 키워드 매칭 → `$contains` 필터 적용 | 예측 가능, 빠름, 무료 | 키워드 목록 관리 필요 |

### 메타데이터 구조 (Week 4 설계)
```python
metadata = {
    "section_num": "2.1.2",                                           # 섹션 번호
    "section_title": "제출서류",                                       # 최하위 제목
    "breadcrumb": "소유권보존등기 > 토지소유권보존등기 > 제출서류",      # 계층 경로
    "start_page": 11,                                                  # 시작 페이지
}
```

In [50]:
!pip install lark

In [51]:
# ── A. LangChain SelfQueryRetriever ──────────────────────────────────────────
# LLM이 질문을 분석해 AttributeInfo 기반 메타데이터 필터를 자동 생성
# Chroma가 해당 필터를 where 절로 변환해 검색 범위를 좁힘

from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo


# breadcrumb 메타데이터 설명: LLM이 올바른 필터를 생성하도록 상세히 기술
metadata_field_info = [
    AttributeInfo(
        name="breadcrumb",
        description=(
            "문서 계층 경로. '대분류 > 중분류 > 소분류' 형식. "
            "대분류 목록: 소유권보존등기, 소유권이전등기, 저당권등기, 전세권등기, 임차권등기, 가등기. "
            "중분류 예시: 토지소유권보존등기, 건물소유권보존등기, 구분건물소유권보존등기, "
            "매매에의한소유권이전등기, 상속에의한소유권이전등기, 증여에의한소유권이전등기, "
            "근저당권설정등기, 전세권설정등기. "
            "소분류 목록: 개념및신청인, 제출서류, 신청서작성, 신청절차. "
            "전체 예시: '소유권보존등기 > 토지소유권보존등기 > 제출서류'"
        ),
        type="string",
    ),
    AttributeInfo(
        name="section_title",
        description=(
            "최하위 섹션 제목. "
            "가능한 값: '개념및신청인', '제출서류', '신청서작성', '신청절차', "
            "'부동산등기의종류및효력', '부동산등기의신청과절차'"
        ),
        type="string",
    ),
    AttributeInfo(
        name="start_page",
        description="섹션 시작 페이지 번호 (정수)",
        type="integer",
    ),
]

document_content_description = (
    "대한민국 부동산 등기 매뉴얼. "
    "소유권보존등기, 소유권이전등기, 저당권등기, 전세권등기, 임차권등기, 가등기 등 "
    "각 등기 유형의 개념, 제출서류, 신청서작성, 신청절차를 포함한다."
)

sq_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=sq_llm,
    vectorstore=db,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True,           # LLM이 생성한 필터를 콘솔에 출력
    search_kwargs={"k": 5},
)

print("A. SelfQueryRetriever 준비 완료")
print("  - LLM(gpt-4o-mini)이 질문 → 메타데이터 필터 자동 생성")
print("  - 생성 필터는 Chroma where 절로 변환되어 검색 범위 제한")
print("  - verbose=True: 실행 시 생성된 필터를 출력")

ImportError: Cannot import lark, please install it with 'pip install lark'.

In [29]:
# ── B. Manual Breadcrumb Filter Retriever ────────────────────────────────────
# 키워드 매칭으로 등기 유형과 섹션 유형을 추출 → Chroma $contains 필터 적용
# SelfQueryRetriever보다 예측 가능하고 빠름. LLM 비용 없음.
#
# Chroma $contains: breadcrumb 문자열에 키워드가 포함된 문서만 검색
# $and: 두 조건을 동시 적용 (등기유형 AND 섹션유형)

MAJOR_CATEGORY_KEYWORDS = {
    "소유권보존등기": ["소유권보존", "보존등기", "보존 등기", "토지보존", "건물보존"],
    "소유권이전등기": ["소유권이전", "이전등기", "매매", "증여", "상속", "경매"],
    "저당권등기":    ["저당권", "근저당", "근저당권", "저당"],
    "전세권등기":    ["전세권"],
    "임차권등기":    ["임차권"],
    "가등기":        ["가등기"],
}

SECTION_TYPE_KEYWORDS = {
    "제출서류":    ["서류", "서류가", "서류는", "필요한 서류", "준비 서류"],
    "신청절차":    ["절차", "과정", "어떻게 신청", "신청하는 방법"],
    "개념및신청인": ["개념", "신청인", "누가 신청", "신청 자격", "신청할 수 있"],
    "신청서작성":  ["신청서", "신청서 작성"],
}


def extract_breadcrumb_filter(query: str) -> dict | None:
    """
    질문에서 등기 유형·섹션 유형을 추출해 Chroma where 필터 생성.

    반환 형식:
      - 두 조건: {"$and": [{"breadcrumb": {"$contains": major}}, {"section_title": {"$eq": section}}]}
      - 등기유형만: {"breadcrumb": {"$contains": major}}
      - 섹션유형만: {"section_title": {"$eq": section}}
      - 감지 불가: None (필터 없이 검색)
    """
    major = None
    section = None

    # 등기 유형 감지: 키워드 매칭 우선, 카테고리명 직접 포함 차선
    for cat, keywords in MAJOR_CATEGORY_KEYWORDS.items():
        if cat in query or any(kw in query for kw in keywords):
            major = cat
            break

    # 섹션 유형 감지
    for sec, keywords in SECTION_TYPE_KEYWORDS.items():
        if sec in query or any(kw in query for kw in keywords):
            section = sec
            break

    if major and section:
        return {
            "$and": [
                {"breadcrumb": {"$contains": major}},
                {"section_title": {"$eq": section}},
            ]
        }
    elif major:
        return {"breadcrumb": {"$contains": major}}
    elif section:
        return {"section_title": {"$eq": section}}
    return None


def manual_filter_retriever(query: str, k: int = 5) -> list:
    """breadcrumb $contains 필터 적용 Dense 검색. 필터 결과 없으면 unfiltered fallback."""
    where_filter = extract_breadcrumb_filter(query)

    if where_filter:
        print(f"  → 적용 필터: {where_filter}")
        results = db.similarity_search(query, k=k, filter=where_filter)
        if results:
            return results
        print(f"  → 필터 결과 없음. Unfiltered fallback 적용.")

    return db.similarity_search(query, k=k)


# 필터 추출 동작 확인
print("=== 필터 추출 테스트 ===")
for q in [
    "토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?",
    "근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?",
    "전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?",
]:
    f = extract_breadcrumb_filter(q)
    print(f"  Q: {q[:40]}...")
    print(f"  → {f}")
    print()

=== 필터 추출 테스트 ===
  Q: 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?...
  → {'$and': [{'breadcrumb': {'$contains': '소유권보존등기'}}, {'section_title': {'$eq': '제출서류'}}]}

  Q: 근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?...
  → {'$and': [{'breadcrumb': {'$contains': '저당권등기'}}, {'section_title': {'$eq': '제출서류'}}]}

  Q: 전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?...
  → {'breadcrumb': {'$contains': '전세권등기'}}



In [30]:
# ── A vs B vs Hybrid+Reranker 비교 ───────────────────────────────────────────

def compare_self_query(query: str, k: int = 3):
    """SelfQueryRetriever / Manual Filter / Hybrid+Reranker 검색 결과 비교"""
    print(f"\n{'='*68}")
    print(f"질문: {query}")
    print(f"{'='*68}")

    # A: SelfQueryRetriever
    print("\n[A. SelfQueryRetriever  (LLM이 breadcrumb 필터 자동 생성)]")
    try:
        sq_docs = self_query_retriever.invoke(query)[:k]
        for i, doc in enumerate(sq_docs, 1):
            bc = doc.metadata.get("breadcrumb", "N/A")
            pg = doc.metadata.get("start_page", "?")
            print(f"  {i}. p.{pg} [{bc}]")
    except Exception as e:
        print(f"  오류 발생: {e}")
        sq_docs = []

    # B: Manual breadcrumb filter
    print("\n[B. Manual BreadcrumbFilter  (키워드 매칭 → $contains 필터)]")
    mf_docs = manual_filter_retriever(query, k=k)
    for i, doc in enumerate(mf_docs, 1):
        bc = doc.metadata.get("breadcrumb", "N/A")
        pg = doc.metadata.get("start_page", "?")
        print(f"  {i}. p.{pg} [{bc}]")

    # C: Hybrid + Reranker (기존 최고 성능, 필터 없음)
    print("\n[C. Hybrid + Re-ranker  (필터 없음, 전체 청크 대상)]")
    hr_docs = hybrid_rerank_retriever(query, top_k=k)
    for i, doc in enumerate(hr_docs, 1):
        bc = doc.metadata.get("breadcrumb", "N/A")
        pg = doc.metadata.get("start_page", "?")
        print(f"  {i}. p.{pg} [{bc}]")


# 테스트 1: 특정 등기유형 + 서류 → 필터링 효과 가장 큼
compare_self_query("토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?")


질문: 토지 소유권 보존등기 신청절차와 필요한 서류가 뭐야?

[A. SelfQueryRetriever  (LLM이 breadcrumb 필터 자동 생성)]
  오류 발생: name 'self_query_retriever' is not defined

[B. Manual BreadcrumbFilter  (키워드 매칭 → $contains 필터)]
  → 적용 필터: {'$and': [{'breadcrumb': {'$contains': '소유권보존등기'}}, {'section_title': {'$eq': '제출서류'}}]}
  → 필터 결과 없음. Unfiltered fallback 적용.
  1. p.10 [소유권보존등기 > 토지소유권보존등기 > 개념및신청인]
  2. p.15 [소유권보존등기 > 토지소유권보존등기 > 신청서작성]
  3. p.11 [소유권보존등기 > 토지소유권보존등기 > 제출서류]

[C. Hybrid + Re-ranker  (필터 없음, 전체 청크 대상)]


KeyboardInterrupt: 

In [ ]:
# 테스트 2: 근저당권 말소 → 저당권등기 필터가 범위를 좁혀야 함
compare_self_query("근저당권 말소등기를 신청하려면 어떤 서류가 필요한가요?")

# 테스트 3: 비교 질문 (필터 감지 어려움) → 필터 없이 fallback 예상
compare_self_query("전세권 설정등기와 임차권 등기의 차이점은 무엇인가요?")

# 테스트 4: 상속 등기 → 소유권이전등기 필터
compare_self_query("상속으로 인한 부동산 등기 신청 시 필요한 서류는 무엇인가요?")

### 9. Error Case 확인

In [47]:
query = "미성년자가 상속으로 부동산을 취득했을 때 등기 신청은 누가 하나요?"

hybrid_docs_ec2 = hybrid_retriever.invoke(query)
reranked_docs_ec2 = hybrid_rerank_retriever(query, top_k=5)

print(f"\n질문: {query}")

print(f"\n[Hybrid + Re-ranker 결과]")
for i, doc in enumerate(reranked_docs_ec2, 1):
    breadcrumb = doc.metadata.get("breadcrumb", "N/A")
    preview = doc.page_content.replace("\n", " ")[:100]
    print(f"  {i}. {breadcrumb}")
    print(f"     {preview}...")


질문: 미성년자가 상속으로 부동산을 취득했을 때 등기 신청은 누가 하나요?

[Hybrid + Re-ranker 결과]
  1. 가등기 > 가등기 > 신청절차
     소유권이전청구권가등기신청절차 ※ 관할등기소는 < 대법원인터넷등기소 - 등기소소개 - 등기소찾기 > 에서확인하실수있습니다 . ※ 제출후접수상황확인 등기소에 신청서를 제출한 후 < 대...
  2. 소유권이전등기 > 상속에의한소유권이전등기 > 신청절차
     . 본인또는자격자대리인이직접등기과 ( 소 ) 를 방문해사용자등록신청을해야합니다 ( 전국등기소어느 곳에서나가능 )[ 「 부동산등기규칙 」 제 68 조제 2 항 ]. ㅇ사용자등록신청시...
  3. 부동산등기의이해 > 부동산등기절차 > 부동산등기의신청과절차
     부동산등기의신청 부동산등기신청의일반원칙 당사자신청주의 ( 「 부동산등기법 」 제 22 조제 1 항 ) 공동신청주의 ( 「 부동산등기법 」 제 23 조제 1 항 ) 공동신청주의의예외...
  4. 소유권이전등기 > 상속에의한소유권이전등기 > 제출서류
     상속에의한소유권이전등기의신청시제출서류 시 · 군 · 구청을통해준비해야하는서류 소유권을증명하는서면 토지대장등본또는임야대장등본 ( 집합 ) 건축물대장등본 신청인의주소및상속을증명하는서면...
  5. 소유권이전등기 > 증여에의한소유권이전등기 > 개념및신청인
     증여에의한소유권이전등기의개념 증여에 의한소유권이전등기는 부동산증여계약에의해소유권을이전하는경우의등기를말합니다 ( 등기 신청안내 , 대법원인터넷등기소 ). 증여의개념 " 증여 " 는당...


In [48]:
# Generation: 검색된 문서 → LLM 답변 생성
answer = get_answer(query, reranked_docs_ec2)

print("\n" + "─" * 60)
print("[LLM 생성 답변]")
print("─" * 60)
print(answer)
print("─" * 60)

print("\n[출처 섹션]")
for i, doc in enumerate(reranked_docs_ec2, 1):
    bc = doc.metadata.get("breadcrumb", "N/A")
    pg = doc.metadata.get("start_page", "?")
    print(f"  {i}. p.{pg} [{bc}]")


────────────────────────────────────────────────────────────
[LLM 생성 답변]
────────────────────────────────────────────────────────────
미성년자가 상속으로 부동산을 취득했을 경우, 등기 신청은 상속인들이 공동으로 신청해야 합니다. 그러나 미성년자는 법적 능력이 제한되므로, 일반적으로 법정대리인인 부모나 후견인이 대신하여 등기 신청을 진행하게 됩니다. 이 경우, 상속인들이 모두 사용자 등록을 신청해야 하며, 필요한 서류를 준비하여 등기소에 제출해야 합니다. 

상속에 의한 소유권 이전 등기는 등기권리자가 단독으로 신청할 수 있지만, 여러 명의 상속인이 있는 경우에는 모두 사용자 등록을 해야 한다는 점을 유의해야 합니다.
────────────────────────────────────────────────────────────

[출처 섹션]
  1. p.194 [가등기 > 가등기 > 신청절차]
  2. p.76 [소유권이전등기 > 상속에의한소유권이전등기 > 신청절차]
  3. p.7 [부동산등기의이해 > 부동산등기절차 > 부동산등기의신청과절차]
  4. p.68 [소유권이전등기 > 상속에의한소유권이전등기 > 제출서류]
  5. p.55 [소유권이전등기 > 증여에의한소유권이전등기 > 개념및신청인]


| # | 질문 유형 | 실패 원인 | 6주차 해결 방향 |
|---|-----------|-----------|------------------|
| EC1 | 비교 질문 (전세권 vs 임차권) | 두 개념을 동시 커버하는 청크 없음 | Query 분해 후 합성 |
| EC2 | 상위 개념 질문 (등기 종류) | 관련 정보가 여러 섹션에 분산 | Router Agent + 섹션 필터링 |
| EC3 | 복합 조건 (미성년자 + 상속) | Multi-hop 정보가 단일 청크에 없음 | Multi-step Agentic RAG |

→  **6주차 Agentic RAG** 도입의 근거

- Query 변환 (HyDE, Multi-Query, Step-Back)
- Router Agent (질문 유형별 검색 전략 분기)
- Multi-step Agentic RAG (ReAct, Tool Use)